# 비물리적지수 구현 결과와 향후 작업 정리

작성일: 2026-09-27

100m 격자별 비물리적지수를 산정하는 코드 생성과 실행까지 완료했다. 물리적지수와 최종 서비스 대상지역 선정은 후속 단계다.

## 1. 확정한 분석 방향

| 항목 | 확정 내용 |
|---|---|
| 목적 | 찾아가는 문화서비스 우선지원지역 선정 |
| 지원 대상 | 60세 이상 취약계층 |
| 대상인구 자료 | 기존 프로젝트의 문화누리 대상 추정인구 |
| 분석 단위 | 100m 격자 |
| 비물리적지수 | 취약노인 규모점수 50% + 커뮤니케이션 부족도 50% |
| 교류 지표 방향 | 교류가 적을수록 지원 필요성이 높음 |
| 1인가구 지표 | 제외 |
| 물리적 분석 출발점 | 각 100m 격자의 대표점. 군집 대표점은 사용하지 않음 |
| 도착점 | 검증된 개별 문화시설 |
| 최종 선정 방식 | 접근성이 나쁜 지역을 먼저 추린 뒤 물리적 50% + 비물리적 50%로 비교(방식 B) |

## 2. 구현한 비물리적지수

### 격자별 취약노인 규모점수

기존 결과에서 각 격자의 60세 이상 장애인·비장애인 문화누리 대상 추정인구를 성별·연령별로 합산한다.

$$T_i=\sum_{s,a\geq60}(\widehat T^{장애}_{isa}+\widehat T^{비장애}_{isa})$$

대상인구가 있는 서울 전체 격자를 대상으로 오름차순 평균순위를 계산한다.

$$D_i=\begin{cases}(r_i-1)/(n-1),&T_i>0\\0,&T_i=0\end{cases}$$

- $n$: 대상인구가 있는 격자 수, $r_i$: 인구수의 평균순위.
- 같은 인구수에는 같은 평균순위를 부여한다.
- 인구 0인 격자는 후보에서 제외한다. 최소 인구 격자의 점수가 0이어도 인구가 있으면 후보에 포함한다.
- 서울 전체를 동일한 기준으로 비교하며 접근성 부족지역 선별 후에도 다시 정규화하지 않는다.
- 순위는 실제 인원수 차이를 보존하지 않으므로 원인구수 $T_i$도 유지한다.

### 행정동별 커뮤니케이션 부족도

구현 기본안으로 평균 통화대상자 수, 평균 문자대상자 수, 평균 통화량, 평균 문자량, SNS 사용횟수의 5개 변수를 각각 20% 반영했다.
같은 성별·연령대끼리 행정동을 비교하며, 값이 낮을수록 높은 부족도를 부여한다.

$$L_{dgk}=1-\frac{r_{dgk}-1}{m_{gk}-1}$$

$$C_{dg}=\frac{1}{5}\sum_{k=1}^{5}L_{dgk},\qquad C_d=\sum_g w_gC_{dg}$$

$d$는 행정동, $g$는 성·연령 집단, $k$는 변수이다. $m_{gk}$는 해당 집단·변수의 유효 행정동 수다.
$w_g$는 같은 통신파일의 총인구수로 계산한 서울 전체 60세 이상 성·연령 구성비이며 모든 동에 동일하게 적용한다.
SNS의 음수는 정상적인 Z-score 값으로 유지한다. 마스킹·결측을 0으로 바꾸지 않는다.

### 격자 연결과 결합

$$C_i=C_{d(i)},\qquad \boxed{N_i=0.5D_i+0.5C_{d(i)}}$$

기존 격자 중심점 기준 행정동 귀속을 사용해 같은 동의 격자에 같은 통신점수를 연결한다.
통신점수는 격자 수로 나누거나 격자 인구비중을 곱하지 않는다.
인구점수는 격자마다 달라 최종 비물리적지수도 격자별로 달라진다.
이는 100m 인구 추정자료와 행정동 통신 특성을 결합한 상대적 우선지원지수이며, 개인의 고립확률이나 격자별 실측 통신점수가 아니다.

## 3. 생성·실행한 노트북과 결과

실행 순서는 01 → 02 → 03이다. 노트북 위치는 프로젝트 기준 `code/03_abstract`이다.

| 노트북 | 수행 내용 |
|---|---|
| `01_senior_population_score.ipynb` | 대상인구 합산, 순위점수 산정, 인구 총량 검증 |
| `02_communication_score.ipynb` | 통신자료 검증, 성·연령별 역순위, 행정동 부족도 산정 |
| `03_grid_nonphysical_index.ipynb` | 행정동 연결, 격자별 지수 결합, 경계 문제 표시, 민감도 검토 |

CSV 출력 경로는 `data/processed/nonphysical_index`이다.

| 결과 파일 | 내용 |
|---|---|
| `grid_senior_population_score.csv` | 격자별 취약노인 규모점수 |
| `dong_communication_score.csv` | 행정동별 통신 부족도 |
| `dong_communication_components.csv` | 성·연령·변수별 계산 내역 |
| `dong_communication_crosswalk.csv` | 행정동 연결표와 대리 적용 표시 |
| `서울시_100m_비물리적지수.csv` | 최종 격자별 비물리적지수 |

### 2026-09-27 실행·검증 결과

| 항목 | 결과 |
|---|---:|
| 전체 격자 | 60,528개 |
| 대상인구가 있는 격자 | 21,263개 |
| 취약노인 추정인구 합계 | 168,748명 |
| 격자 ID 중복 | 0개 |
| 대리 적용 포함 비물리적지수 결측 | 0개 |
| 기존 인구 총량 | 보존 확인 |

SNS 제외 전후와 인구·통신 가중치 변경에 따른 순위 변화도 검토했다. 실제 저장 점수는 확정한 50:50을 유지했다.
이 표는 위 날짜의 실행 기록이며, 입력 갱신 후에는 아래 셀과 후속 노트북 출력으로 최신 수치를 확인한다.

## 4. 현재 결과의 제한과 추가 확인 사항

### 행정동 경계 대리 적용

| 현재 행정동 | 연결한 통신자료 | 처리 |
|---|---|---|
| 개포3동 | 일원2동 | 명칭 변경 |
| 상일1동 | 상일동 | 명칭 변경 |
| 상일2동 | 강일동 | 분동 전 관할 동 점수 대리 적용 |
| 항동 | 오류2동 | 분동 전 관할 동 점수 대리 적용 |

강일동·오류2동도 통신자료가 분동 전 경계를 나타낼 가능성이 있어 함께 대리 적용으로 표시했다.
총 798개 격자가 해당하며, 그중 대상인구가 있는 곳은 292개 격자·2,275명이다.
전체 점수와 별도로 `비물리적지수_대리적용제외`를 만들고 해당 지역은 결측으로 남겼다.
공식 연혁은 확인했지만 통신파일의 실제 집계 경계까지 확인한 것은 아니므로 전체 결과는 탐색적 결과다.

통신자료의 최상위 연령코드 75는 공식 안내를 근거로 75세 이상으로 해석해 구현했다. 세부 코드 정의는 추가 확인이 필요하다.
통신변수의 제공 평균은 매뉴얼의 집계 대상 정의를 따르며 전체 주민의 무조건부 평균으로 단정하지 않는다.
기초생활수급자 2023년, 차상위 2021년, 일반 인구 2024년, 통신 2025년 12월 자료로 기준시점이 혼재한다.
인구 자체도 일반 인구의 구성비 등을 적용한 추정치이므로 특정 시점의 실측 취약노인 인구로 해석하지 않는다.

### 저장 경로 문서화

출력은 `data/processed` 저장 원칙과 프로젝트 루트 상대경로 규칙을 따른다.
다만 새 하위 폴더 `nonphysical_index`와 새 분석 단계는 `AGENTS.md`·README·메타데이터에 아직 반영되지 않았다.

## 5. 앞으로 필요한 작업

| 순서 | 필요한 작업 | 목적 |
|---|---|---|
| 1 | 통신 행정동 집계 경계, 연령코드, 변수 집계 대상 확인 | 대리 적용과 해석상의 불확실성 줄이기 |
| 2 | `AGENTS.md`·README·메타데이터에 새 단계와 결과 경로 반영 | 코드와 문서 구조 일치 |
| 3 | 시설 운영 여부, 중복, 공연·전시·스포츠 분류, 출입구 좌표 확인 | 실제 이용 가능한 도착지 확정 |
| 4 | OSM 보행망, DEM, 노선·배차·운행시간 확보 | 거리·경사·환승·대기 부담 계산 |
| 5 | 각 격자에서 시설까지 경로 계산 및 물리적 부족도 생성 | $P_i$ 산정 |
| 6 | 접근성 부족도 기준과 후보 규모 비교 | 우선지원 후보지역 설정 |
| 7 | 최종점수 계산, 인접 격자 묶기, 운영 장소·수용인원 검토 | 실제 방문서비스 지역 선정 |

물리적지수는 이동시간 정규화 기준, 시설 접근시간 한계, 경사·계단 처리, 접근성 감소계수 $\beta$를 정해야 한다.
접근성이 나쁠수록 $P_i$가 높아지도록 방향을 통일한다.

$$\mathcal S=\{i:T_i>0,\ P_i\geq\tau\}$$

$$\boxed{F_i=0.5P_i+0.5N_i=0.50P_i+0.25D_i+0.25C_{d(i)}}\qquad(i\in\mathcal S)$$

후보 선별 기준 $\tau$는 아직 미정이다. 현재 저장된 순위는 **비물리적 지원 필요성 순위**이며 물리적지수와 최종 선정순위는 아직 계산하지 않았다.

---


# 01. 60세 이상 취약계층 규모점수

## 논의 결과와 실행 순서
1. 지원 대상은 60세 이상 문화누리 대상 **추정인구**이다. 전체 노인인구를 다시 더하지 않는다.
2. 비물리적지수는 $N_i=0.5D_i+0.5C_{d(i)}$이며 1인가구 지표는 제외한다.
3. $D_i$는 서울 전체의 대상인구가 있는 100m 격자끼리 비교한 평균순위 점수이다.
4. $C_d$는 성·연령별로 비교한 5개 통신변수의 부족도를 공통 인구 구성비로 통합한다.
5. 통신점수는 기존 **격자 중심점 기준 행정동 귀속**에 맞춰 동일 부여한다.
6. 물리적지수는 각 100m 격자에서 개별 시설까지 직접 계산한다. 군집 출발점은 사용하지 않는다.
7. 향후 접근성 부족지역을 먼저 선별한 뒤 $F_i=0.5P_i+0.5N_i$로 비교한다(방식 B).
8. 접근성 선별 기준, 이동시간 한계·정규화, 경로·시설 검증은 후속 과제다. 이 단계에서는 물리적·최종지수를 생성하지 않는다.

실행 순서: 01 → 02 → 03. 결과는 `data/processed/nonphysical_index`에 저장한다.
기존 인구 노트북과 원자료는 수정하지 않는다. 같은 버전의 입력으로 재실행하면 본 단계 결과만 갱신된다.

## 인구점수
$$T_i=\sum_{s,a\geq60}(\widehat T^{장애}_{isa}+\widehat T^{비장애}_{isa})$$
$$D_i=(r_i-1)/(n-1)\quad(T_i>0),\qquad D_i=0\quad(T_i=0)$$
$n$은 대상인구가 있는 격자 수, $r_i$는 서울 전체 오름차순 평균순위다.
동률은 평균순위로 처리한다. 최소 인구 격자의 점수가 0이어도 인구가 있으면 분석대상이다.
인구 0은 후보 제외이며 이후 후보 선별 후에도 다시 정규화하지 않는다.

## 해석상 한계
기초생활수급자 2023년·차상위 2021년·일반 인구 2024년·통신 2025년 12월 자료를 결합한다.
격자 취약노인 수는 일반 인구의 성·연령 구성 등을 적용한 추정값이지 실측 명부가 아니다.
순위는 실제 인원수 격차를 보존하지 않으므로 원인구수를 반드시 유지한다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [start, *start.parents]
                    if (p / 'data/raw').is_dir() and (p / 'code').is_dir())
OUT = PROJECT_ROOT / 'data/processed/nonphysical_index'
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

def rank01(values):
    # 결측을 정상 점수로 바꾸지 않는다. 비교대상이 하나면 순위를 정의할 수 없다.
    assert values.notna().all() and np.isfinite(values).all()
    assert len(values) > 1, '순위 비교대상이 2개 미만입니다.'
    return (values.rank(method='average') - 1) / (len(values) - 1)

def save_checked(frame, name, keys):
    path = OUT / name
    assert not frame.duplicated(keys).any()
    frame.to_csv(path, index=False, encoding='utf-8-sig')
    saved = pd.read_csv(path, encoding='utf-8-sig')
    assert len(saved) == len(frame) and not saved.duplicated(keys).any()
    for col in frame.select_dtypes(include='number'):
        assert np.allclose(frame[col], saved[col], equal_nan=True), col
    print('저장 및 재검증:', path.relative_to(PROJECT_ROOT))
    return saved

## 1. 기존 60세 이상 인구 결과 확인

In [2]:
source = PROJECT_ROOT / 'data/processed/estimated_target_population/서울시_격자_100m_문화누리대상자_60세이상_성연령장애별_인구수.csv'
population = pd.read_csv(source, encoding='utf-8-sig')
pop_cols = ['문화누리대상자_성연령장애별_추정_인구수', '문화누리대상자_성연령비장애별_추정_인구수']
keys = ['GRID_CD', '시군구', '행정동']
assert population[keys + ['성별','연령대'] + pop_cols].notna().all().all()
assert not population.duplicated(['GRID_CD','성별','연령대']).any()
assert set(population['연령대']) == {'60-69세','70-79세','80-89세','90-99세','100세-'}
assert population[pop_cols].ge(0).all().all()
assert np.allclose(population[pop_cols], np.round(population[pop_cols]))
assert population.groupby('GRID_CD')[['시군구','행정동']].nunique().eq(1).all().all()
display(population.groupby('연령대')[pop_cols].sum())

,문화누리대상자_성연령장애별_추정_인구수,문화누리대상자_성연령비장애별_추정_인구수
연령대,,
100세-,0,0
60-69세,5848,84334
70-79세,1087,54233
80-89세,257,22226
90-99세,18,745


## 2. 격자별 합산과 서울 전체 순위

In [3]:
grid = population.groupby(keys, as_index=False)[pop_cols].sum()
grid['취약노인수'] = grid[pop_cols].sum(axis=1)
grid['대상인구존재'] = grid['취약노인수'].gt(0)
positive = grid['대상인구존재']
grid['인구규모점수'] = 0.0
grid.loc[positive, '인구규모점수'] = rank01(grid.loc[positive, '취약노인수'])
assert grid['취약노인수'].sum() == population[pop_cols].sum().sum()
assert grid['인구규모점수'].between(0,1).all()
assert grid.groupby('취약노인수')['인구규모점수'].nunique().eq(1).all()
assert grid.loc[positive].sort_values('취약노인수')['인구규모점수'].is_monotonic_increasing
assert np.allclose(rank01(pd.Series([1,2,2,5,10])), [0,.375,.375,.75,1])
display(pd.Series({'격자 수':len(grid), '대상인구 있는 격자':int(positive.sum()),
                   '취약노인 합계':int(grid['취약노인수'].sum())}).to_frame('값'))
display(grid.loc[positive, ['취약노인수','인구규모점수']].describe())

,값
격자 수,60528
대상인구 있는 격자,21263
취약노인 합계,168748


,취약노인수,인구규모점수
count,21263.000000,21263.000000
mean,7.936227,0.500000
std,9.082616,0.287806
min,1.000000,0.055663
25%,3.000000,0.276526
50%,6.000000,0.523751
75%,10.000000,0.734997
max,310.000000,1.000000


## 3. 저장 및 총량 재검증

In [4]:
saved = save_checked(grid, 'grid_senior_population_score.csv', ['GRID_CD'])
assert saved['취약노인수'].sum() == population[pop_cols].sum().sum()
assert saved.isna().sum().sum() == 0

저장 및 재검증: data\processed\nonphysical_index\grid_senior_population_score.csv
